# 🧠 LAB 09:Philosophy RAG Workshop: From Raw Data to Knowledge Graphs


In this **final workshop**, you'll compare three ways to use LLMs:

1. **Plain LLM**: Just the API (what you already know)
2. **RAG (Retrieval-Augmented Generation)**: LLM + Academic Papers
3. **GraphRAG**: LLM + Knowledge Relationships

**By the end**, you'll see how each approach gives **different results** for the same philosophical question!

---

## ⏱️ **Time Guide** (Total: ~90-120 minutes)
1. **Setup** (5-8 min) - Install packages & API setup
2. **Dataset Loading** (8-10 min) - Explore philosophy papers  
3. **Plain LLM** (8-12 min) - Simple API calls + model comparison
4. **RAG System** (15-20 min) - Vector search + LLM context
5. **Knowledge Graphs** (12-15 min) - Build relationships + extract triplets
6. **GraphRAG** (10-15 min) - Combined graph + retrieval approach
7. **Three-way Comparison** (15-20 min) - See all approaches on same question
8. **Reflection & Discussion** (8-10 min) - Decision frameworks + next steps

💡 **Tip**: Work at your own pace! Each section builds on the previous one.

---

## 🎯 Learning Goals
- **What is RAG?** (and why is it useful for research)
- **What is a Vector Database?** (and how it finds relevant papers)
- **What is a Knowledge Graph?** (and how it connects ideas)
- **What is GraphRAG?** (combining graphs with retrieval)
- **When to use each approach** (practical decision-making)


## 🔧 Part 1: Setup (10 minutes)

### Install Required Libraries

In [ ]:
# Install everything we need
%pip install -q datasets sentence-transformers faiss-cpu networkx openai requests pandas numpy

print("📦 All packages installed successfully!")
print("✅ Ready for the workshop!")


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import requests
import json
import time
import re
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter

# ML and embeddings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

# Graph analysis
import networkx as nx

# LLM integration
from openai import OpenAI
from google.colab import userdata

print("📚 All libraries imported successfully!")



### 🔑 API Setup

**For this workshop, you'll use the same API setup from the previous session.**


In [ ]:
# === API Setup (same as previous session) ===

### OPTION 1: HuggingFace Setup (Uncomment to use)
# HUGGINGFACE_API_KEY = userdata.get('HUGGINGFACE_API_KEY')  # For HuggingFace
# API_PROVIDER = "huggingface"

### OPTION 2: OpenRouter Setup (Currently active)
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')    # For OpenRouter
API_PROVIDER = "openrouter"

print("🔐 Using Colab secrets for API keys")
print(f"✅ {API_PROVIDER} selected")


In [ ]:
# 🎯 Multiple Models with Smart Fallback
# New concept: If one model fails, try another automatically!

OPENROUTER_MODELS = [
    "microsoft/phi-3-mini-128k-instruct:free",
    "meta-llama/llama-3.2-3b-instruct:free", 
    "qwen/qwen-2-7b-instruct:free",
    "anthropic/claude-3-haiku"
]

HUGGINGFACE_MODELS = [
    "microsoft/DialoGPT-medium",
    "microsoft/DialoGPT-large",
    "meta-llama/Llama-2-7b-chat-hf"
]

def call_llm_with_fallback(prompt: str, max_tokens: int = 200, preferred_model: str = None) -> dict:
    """Try multiple models until one works - smart fallback!"""
    
    if API_PROVIDER == "openrouter":
        models_to_try = OPENROUTER_MODELS.copy()
        api_key = OPENROUTER_API_KEY
    else:
        models_to_try = HUGGINGFACE_MODELS.copy()
        api_key = HUGGINGFACE_API_KEY
    
    # If student wants a specific model, try it first
    if preferred_model and preferred_model in models_to_try:
        models_to_try = [preferred_model] + [m for m in models_to_try if m != preferred_model]
    
    print(f"🔄 Trying {API_PROVIDER} models: {len(models_to_try)} available")
    
    for model in models_to_try:
        try:
            print(f"   🤖 Trying: {model.split('/')[-1]}...")
            
            if API_PROVIDER == "openrouter":
                response = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={
                        "Authorization": f"Bearer {api_key}",
                        "Content-Type": "application/json"
                    },
                    json={
                        "model": model,
                        "messages": [{"role": "user", "content": prompt}],
                        "max_tokens": max_tokens
                    },
                    timeout=30
                )
                
                if response.status_code == 200:
                    result = response.json()["choices"][0]["message"]["content"]
                    print(f"   ✅ Success!")
                    return {
                        "response": result,
                        "model_used": model.split('/')[-1],
                        "success": True,
                        "error": None
                    }
                    
            else:  # huggingface
                response = requests.post(
                    f"https://api-inference.huggingface.co/models/{model}",
                    headers={"Authorization": f"Bearer {api_key}"},
                    json={"inputs": prompt, "parameters": {"max_new_tokens": max_tokens}},
                    timeout=30
                )
                
                if response.status_code == 200:
                    result = response.json()[0]["generated_text"]
                    print(f"   ✅ Success!")
                    return {
                        "response": result,
                        "model_used": model.split('/')[-1],
                        "success": True,
                        "error": None
                    }
            
            print(f"   ❌ Failed")
                
        except Exception as e:
            print(f"   ❌ Error")
            continue
    
    # If all failed
    return {
        "response": "All models unavailable. Try again later.",
        "model_used": "none",
        "success": False,
        "error": "All models failed"
    }

# Simple wrapper for backward compatibility
def call_llm(prompt: str, max_tokens: int = 200) -> str:
    """Simple wrapper that just returns the response text"""
    result = call_llm_with_fallback(prompt, max_tokens)
    return result["response"]

# Test the fallback mechanism
print("🧪 Testing fallback mechanism...")
test_result = call_llm_with_fallback("What is philosophy?")
print(f"\n📊 Test Results:")
print(f"   ✅ Success: {test_result['success']}")
print(f"   🤖 Model used: {test_result['model_used']}")
print(f"   💬 Response: {test_result['response'][:100]}...")

print("\n✅ Fallback system ready!")


In [ ]:
# Load the philosophy papers dataset
print("📥 Loading philosophy papers dataset...")
dataset = load_dataset("maximuspowers/philpapers-papers-summarized-labeled")
df = dataset['train'].to_pandas()

print(f"✅ Loaded {len(df)} philosophy papers")
print(f"📊 Columns: {list(df.columns)}")

# Look at a sample paper
sample_paper = df.iloc[0]
print(f"\n📄 Sample Paper:")
print(f"Title: {sample_paper['title']}")
print(f"Summary: {sample_paper['summary'][:200]}...")
print(f"Philosophy Schools: {sample_paper['philosophy_schools']}")


### 🔍 TODO 3: Discover Philosophical Schools (3-5 minutes)

**Your task**: Complete the code to find all philosophical schools in the dataset.

**What you'll learn**: How to explore real research data.


In [ ]:
# === TODO 3: Find all unique philosophical schools ===

def extract_all_schools(df):
    """Extract all unique philosophical schools from the dataset"""
    all_schools = []
    for schools_array in df['philosophy_schools']:
        # Fill in the blank: Check if schools_array is iterable (has __iter__) and not a string
        if hasattr(schools_array, '__________') and not isinstance(schools_array, str):  # Fill in: '__iter__'
            # Fill in the blank: Convert to list and extend all_schools
            all_schools.extend(________(schools_array))  # Fill in: list
    # Fill in the blank: Return sorted unique schools
    return sorted(list(set(__________)))  # Fill in: all_schools

# Fill in the blank: call the function with the dataframe
unique_schools = __________(df)  # Fill in: extract_all_schools

print(f"📚 Total philosophical schools: {len(unique_schools)}")
print(f"🎯 Schools found: {unique_schools}")

# Create a column for the primary school of each paper
df['primary_school'] = df['philosophy_schools'].apply(
    lambda x: x[0] if hasattr(x, '__len__') and len(x) > 0 else 'Unknown'
)

# Show distribution
school_counts = df['primary_school'].value_counts()
print(f"\n📊 Top 5 schools by paper count:")
for school, count in school_counts.head(5).items():
    print(f"  {school}: {count} papers")

print("\n✅ Dataset exploration complete!")


## 🤖 Part 3: Approach 1 - Plain LLM (What You Already Know) (8-12 minutes)

Let's start with what you learned in the previous session: **just asking an LLM directly**.


In [ ]:
def ask_plain_llm(question: str, preferred_model: str = None) -> dict:
    """Ask a question using just the LLM (no additional context)"""
    prompt = f"""You are a helpful philosophy assistant. Answer this question clearly and concisely:

Question: {question}

Answer:"""
    
    result = call_llm_with_fallback(prompt, preferred_model=preferred_model)
    return result

# === TODO 3.1: Try Different Models for Plain LLM ===
print("🎯 Testing Plain LLM with different models...")

# Test with a philosophical question
test_question = "What is the relationship between consciousness and free will?"
print(f"❓ Question: {test_question}")

# Students can experiment with different models
print(f"\n🤖 PLAIN LLM - Default Model:")
result1 = ask_plain_llm(test_question)
print(f"Model used: {result1['model_used']}")
print(f"Response: {result1['response']}")

# Show how to specify a preferred model
print(f"\n🤖 PLAIN LLM - Trying a Different Model:")
if API_PROVIDER == "openrouter":
    preferred = "meta-llama/llama-3.2-3b-instruct:free"
else:
    preferred = "microsoft/DialoGPT-large"

result2 = ask_plain_llm(test_question, preferred_model=preferred)
print(f"Preferred model: {preferred}")
print(f"Actually used: {result2['model_used']}")
print(f"Response: {result2['response']}")

print("\n💡 Notice: Different models might give different perspectives on the same question!")
print("✅ Plain LLM approach with model selection working!")


## 🔍 Part 4: Approach 2 - RAG (Retrieval-Augmented Generation) (15-20 minutes)

### 🎯 What is RAG?

**RAG = Retrieval-Augmented Generation**

Instead of just asking the LLM, we:
1. **Find relevant papers** from our dataset
2. **Give those papers to the LLM** as context
3. **Ask the LLM to answer** based on the papers

**Why is this better?**
- ✅ **Grounded in real research**: Answers cite actual papers
- ✅ **More accurate**: Based on scholarly sources  
- ✅ **Verifiable**: You can check the source papers
- ✅ **Domain-specific**: Focused on philosophy research


In [ ]:
# Create a diagram to explain RAG
print("""🎯 RAG System Overview:

┌─────────────┐    ┌─────────────────┐    ┌─────────────────┐    ┌─────────────┐
│   Question  │───▶│  Vector Search  │───▶│ Relevant Papers │───▶│ LLM + Context│
│             │    │                 │    │                 │    │             │
│"What is     │    │ Find papers     │    │ 3-5 most        │    │ Answer with │
│consciousness│    │ similar to      │    │ relevant papers │    │ citations   │
│?"           │    │ the question    │    │ about topic     │    │             │
└─────────────┘    └─────────────────┘    └─────────────────┘    └─────────────┘

🔑 Key Innovation: LLM gets REAL PAPERS as context, not just its training memory!
""")


### 🧮 What is a Vector Database?

**Vector Database** = A way to find "similar" text using mathematics

**How it works:**
1. **Convert text to numbers** (vectors): Each paper becomes a list of numbers
2. **Store all vectors**: Database of number lists
3. **Find similar vectors**: Papers with similar numbers = similar meaning

**Example:**
- "consciousness" → [0.2, 0.8, 0.1, ...]
- "awareness" → [0.3, 0.7, 0.2, ...]  ← Similar numbers!
- "pizza" → [0.9, 0.1, 0.5, ...]      ← Very different numbers


In [ ]:
# Step 1: Create a vector database for our philosophy papers
print("🧮 Building vector database...")

# Load a model that converts text to vectors (numbers)
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("✅ Text-to-vector model loaded")

# Convert all paper summaries to vectors
print("📊 Converting papers to vectors...")
paper_vectors = embedder.encode(df['summary'].tolist())
print(f"✅ Created {len(paper_vectors)} vectors, each with {paper_vectors.shape[1]} dimensions")

# Create a fast search index (FAISS = Facebook AI Similarity Search)
vector_index = faiss.IndexFlatIP(paper_vectors.shape[1])  # IP = Inner Product (similarity)
faiss.normalize_L2(paper_vectors)  # Normalize for better similarity matching
vector_index.add(paper_vectors.astype('float32'))

print(f"✅ Vector database ready with {vector_index.ntotal} papers!")
print("🔍 Can now find similar papers in milliseconds!")


### 🔍 TODO 4: Build a Paper Search Function (5-8 minutes)

**Your task**: Complete the function to find relevant papers.

**What you'll learn**: How vector similarity search works.


In [ ]:
# === TODO 4: Complete the paper search function ===

def find_relevant_papers(question: str, num_papers: int = 3) -> List[Dict]:
    """Find papers most relevant to a question"""
    
    # Step 1: Convert question to vector
    question_vector = embedder.encode([question])
    faiss.normalize_L2(question_vector)
    
    # Step 2: Search for similar papers
    scores, indices = vector_index.search(question_vector.astype('float32'), num_papers)
    
    # Step 3: Get paper information
    relevant_papers = []
    for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
        paper = df.iloc[idx]
        
        # Fill in the blank: create paper dictionary
        paper_info = {
            'title': paper['title'],           # Fill in: paper['title']
            'summary': paper['summary'],         # Fill in: paper['summary']
            'school': paper['primary_school'],          # Fill in: paper['primary_school']
            'similarity': float(score),
            'rank': i + 1
        }
        relevant_papers.append(paper_info)
    
    return relevant_papers

# Test the search function
test_question = "What is consciousness?"
papers = find_relevant_papers(test_question)

print(f"🔍 Search results for: '{test_question}'")
for paper in papers:
    print(f"\n{paper['rank']}. {paper['title']} (similarity: {paper['similarity']:.3f})")
    print(f"   School: {paper['school']}")
    print(f"   Summary: {paper['summary'][:100]}...")

print("\n✅ Paper search working!")


In [ ]:
# === TODO 5: Complete the RAG system ===

def ask_with_rag(question: str, preferred_model: str = None) -> dict:
    """Answer a question using RAG (search + LLM)"""
    
    # Step 1: Find relevant papers
    papers = find_relevant_papers(question)  # Fill in: find_relevant_papers
    
    # Step 2: Format papers as context for the LLM
    context = "Based on these philosophy papers:\n\n"
    
    for paper in papers:
        # Fill in the blank: add paper to context
        context += f"Paper {paper['rank']}: {paper['title']}\n"  # Fill in: paper['title']
        context += f"School: {paper['school']}\n"                   # Fill in: paper['school']
        context += f"Summary: {paper['summary'][:200]}...\n\n"
    
    # Step 3: Create prompt for LLM
    prompt = f"""{context}
Question: {question}

Please answer the question based on the papers above. Include citations (e.g., "according to Paper 1"):

Answer:"""
    
    # Step 4: Get LLM response with model selection
    result = call_llm_with_fallback(prompt, max_tokens=300, preferred_model=preferred_model)
    
    # Add paper information to the result
    result['papers_used'] = papers
    result['num_papers'] = len(papers)
    
    return result

# === TODO 5.1: Compare Models in RAG Context ===
print("🎯 Testing RAG with different models...")

# Test the complete RAG system
test_question = "What is the relationship between consciousness and free will?"
print(f"❓ Question: {test_question}")

print(f"\n🔍 RAG - Default Model:")
rag_result1 = ask_with_rag(test_question)
print(f"Papers found: {rag_result1['num_papers']}")
print(f"Model used: {rag_result1['model_used']}")
print(f"Response: {rag_result1['response']}")

print(f"\n🔍 RAG - Different Model:")
# Try a different model for comparison
if API_PROVIDER == "openrouter":
    preferred = "qwen/qwen-2-7b-instruct:free"
else:
    preferred = "meta-llama/Llama-2-7b-chat-hf"

rag_result2 = ask_with_rag(test_question, preferred_model=preferred)
print(f"Preferred model: {preferred}")
print(f"Actually used: {rag_result2['model_used']}")
print(f"Response: {rag_result2['response']}")

print("\n💡 Notice: Same papers, different models = different interpretations!")
print("✅ RAG system with model selection working!")


## 🌐 Part 5: Knowledge Graphs - Connecting Ideas (12-15 minutes)

### 🧠 What is a Knowledge Graph?

A **Knowledge Graph** is a network of facts, where each fact is a "triple":
- **(Subject, Relation, Object)**

**Examples from philosophy:**
- ("Socrates", "is a", "Philosopher")
- ("Utilitarianism", "influences", "Effective Altruism")
- ("Free Will", "relates to", "Moral Responsibility")


In [ ]:
# Diagram showing knowledge graph structure
print("""🌐 Knowledge Graph Structure:

    ┌─────────────┐      belongs to      ┌─────────────┐
    │ Paper Title │────────────────────▶│Utilitarianism│
    └─────────────┘                     └─────────────┘
           │                                   │
           │ has                           co-occurs with
           ▼                                   ▼
    ┌─────────────┐                     ┌─────────────┐
    │    Link     │                     │Existentialism│
    └─────────────┘                     └─────────────┘
                                              │
                                        mentions concept
                                              ▼
                                      ┌─────────────┐
                                      │Consciousness│
                                      └─────────────┘

🔑 Key: Instead of just finding similar text, we can follow RELATIONSHIPS!
""")


### 📊 TODO 6: Extract Knowledge Triplets from Our Dataset (5-8 minutes)

**Your task**: Help extract simple relationships from our philosophy papers.

**What you'll learn**: How to build knowledge graphs from real data.


In [ ]:
# === Create a simple knowledge graph from our papers ===

import networkx as nx

# Create an empty graph
knowledge_graph = nx.Graph()

print("🌐 Building knowledge graph from philosophy papers...")

# Extract triplets from a few sample papers
sample_papers = df.head(10)  # Just first 10 papers for demo

triplets = []  # Store all our knowledge triplets

for idx, paper in sample_papers.iterrows():
    title = paper['title']
    link = paper['link']
    schools = paper['philosophy_schools']
    
    # Type 1: Paper → belongs to school → School
    for school in schools:
        triplet = (title, "belongs to school", school)
        triplets.append(triplet)
        knowledge_graph.add_edge(title, school, relation="belongs_to")
    
    # Type 2: Paper → has → link
    triplet = (title, "has", link)
    triplets.append(triplet)
    knowledge_graph.add_edge(title, "URL", relation="has_link")
    
    # Type 3: School → co-occurs with → School (if both in same paper)
    for i, school1 in enumerate(schools):
        for school2 in schools[i+1:]:
            triplet = (school1, "co-occurs with", school2)
            triplets.append(triplet)
            knowledge_graph.add_edge(school1, school2, relation="co_occurs")

# Show some example triplets
print(f"\n📊 Generated {len(triplets)} knowledge triplets")
print("\n🔍 Example triplets:")
for i, (subj, rel, obj) in enumerate(triplets[:5]):
    print(f"{i+1}. ({subj[:30]}{'...' if len(subj) > 30 else ''}, {rel}, {obj[:30]}{'...' if len(obj) > 30 else ''})")

print(f"\n🌐 Knowledge graph created:")
print(f"   📊 Nodes: {knowledge_graph.number_of_nodes()}")
print(f"   🔗 Edges: {knowledge_graph.number_of_edges()}")


In [ ]:
# === TODO 7: Extract a concept mention triplet ===

# Look at this paper summary
sample_paper = df.iloc[0]
print(f"📄 Paper: {sample_paper['title']}")
print(f"📝 Summary: {sample_paper['summary']}")
print(f"🏫 Schools: {sample_paper['philosophy_schools']}")

print("\n🔍 Look at the summary above. Can you find a philosophical concept mentioned?")
print("Examples: consciousness, free will, justice, truth, reality, etc.")

# === Your turn! ===
# Fill in the blank: What concept do you see mentioned?
concept_you_found = "___________"  # Fill in a concept from the summary

# Create your triplet
if concept_you_found != "___________":
    your_triplet = (sample_paper['title'], "mentions concept", concept_you_found)
    print(f"\n✅ Your triplet: {your_triplet}")
    
    # Add it to the knowledge graph
    knowledge_graph.add_edge(sample_paper['title'], concept_you_found, relation="mentions")
    print(f"🌐 Added to knowledge graph!")
    print(f"   📊 Graph now has {knowledge_graph.number_of_nodes()} nodes and {knowledge_graph.number_of_edges()} edges")
else:
    print("\n❓ Fill in a concept you found in the summary above!")

print("\n💡 What other relationships could you add? Think about:")
print("   - (Author, wrote, Paper)")
print("   - (Concept, opposes, Another Concept)")
print("   - (School, influenced by, Historical Figure)")
print("   - (Argument, supports, Conclusion)")


## 🚀 Part 6: Approach 3 - GraphRAG (Knowledge Graph + Retrieval) (10-15 minutes)

### 🎯 What is GraphRAG?

**GraphRAG** combines the best of both worlds:
1. **Vector search** finds papers similar to your question
2. **Graph traversal** finds related concepts through relationships
3. **LLM** answers using both types of information

**Why is this powerful?**
- ✅ **Discovers connections**: Finds relationships you might miss
- ✅ **Broader context**: Not just direct matches
- ✅ **Serendipity**: Unexpected but relevant connections


In [ ]:
# Simple GraphRAG implementation
def ask_with_graph_rag(question: str, preferred_model: str = None) -> dict:
    """Answer using both vector search AND knowledge graph"""
    
    # Step 1: Get papers from vector search (like regular RAG)
    vector_papers = find_relevant_papers(question, num_papers=2)
    
    # Step 2: Extract concepts from the question
    question_concepts = []
    for school in unique_schools:
        if school.lower() in question.lower():
            question_concepts.append(school)
    
    # Step 3: Find related concepts in the knowledge graph
    related_concepts = []
    for concept in question_concepts:
        if knowledge_graph.has_node(concept):
            # Get neighbors (related concepts)
            neighbors = list(knowledge_graph.neighbors(concept))
            related_concepts.extend(neighbors[:3])  # Max 3 related concepts
    
    # Step 4: Build enhanced context
    context = "Based on vector search and knowledge graph:\n\n"
    
    # Add vector search results
    context += "📊 Most relevant papers:\n"
    for paper in vector_papers:
        context += f"- {paper['title']} (School: {paper['school']})\n"
        context += f"  {paper['summary'][:150]}...\n\n"
    
    # Add graph relationships
    if related_concepts:
        context += "🌐 Related concepts from knowledge graph:\n"
        for concept in related_concepts[:5]:  # Limit to avoid too much text
            context += f"- {concept}\n"
        context += "\n"
    
    # Step 5: Get LLM response with model selection
    prompt = f"""{context}
Question: {question}

Please answer using both the papers and related concepts above:

Answer:"""
    
    result = call_llm_with_fallback(prompt, max_tokens=300, preferred_model=preferred_model)
    
    # Add graph information to the result
    result['papers_used'] = vector_papers
    result['concepts_found'] = question_concepts
    result['related_concepts'] = related_concepts
    
    return result

# === TODO 6.1: Compare Models in GraphRAG Context ===
print("🎯 Testing GraphRAG with different models...")

# Test GraphRAG
test_question = "How does utilitarianism relate to ethics?"
print(f"❓ Question: {test_question}")

print(f"\n🌐 GraphRAG - Default Model:")
graph_result1 = ask_with_graph_rag(test_question)
print(f"Papers: {len(graph_result1['papers_used'])}, Concepts: {len(graph_result1['related_concepts'])}")
print(f"Model used: {graph_result1['model_used']}")
print(f"Response: {graph_result1['response']}")

print(f"\n🌐 GraphRAG - Different Model:")
# Try another model
if API_PROVIDER == "openrouter":
    preferred = "microsoft/phi-3-mini-128k-instruct:free"
else:
    preferred = "microsoft/DialoGPT-medium"

graph_result2 = ask_with_graph_rag(test_question, preferred_model=preferred)
print(f"Preferred model: {preferred}")
print(f"Actually used: {graph_result2['model_used']}")
print(f"Response: {graph_result2['response']}")

print("\n💡 Notice: Same context (papers + graph), different reasoning styles!")
print("✅ GraphRAG system with model selection working!")


## 🏆 Part 7: The Big Comparison - All Three Approaches (15-20 minutes)

**Now let's see how all three approaches answer the SAME question differently!**

### 🎯 TODO 8: Compare All Three Approaches

**Your task**: Choose a philosophical question and see how each approach responds.

**What you'll learn**: When to use each approach for research.


In [ ]:
# === TODO 8: Choose your question and compare approaches ===

# Fill in the blank: Choose a philosophical question that interests you
your_question = "___________"  # Fill in: e.g., "What is the meaning of life?"

# === TODO 8.1: Choose your preferred model (optional) ===
# Fill in the blank: Choose a specific model to try, or leave as None for default fallback
your_preferred_model = None  # Fill in: e.g., "meta-llama/llama-3.2-3b-instruct:free" or None

# Default question if you haven't filled it in
if your_question == "___________":
    your_question = "What is the relationship between consciousness and identity?"

print(f"🎯 Comparing all approaches for: '{your_question}'")
if your_preferred_model:
    print(f"🤖 Preferred model: {your_preferred_model}")
else:
    print(f"🤖 Using fallback mechanism (tries models in order)")
print("=" * 80)

# Approach 1: Plain LLM
print("\n🤖 APPROACH 1: Plain LLM (just the API)")
print("-" * 50)
plain_result = ask_plain_llm(your_question, preferred_model=your_preferred_model)
print(f"Model used: {plain_result['model_used']}")
print(f"Success: {plain_result['success']}")
print(f"Response: {plain_result['response']}")

print("\n" + "=" * 80)

# Approach 2: RAG (LLM + Papers)
print("\n📚 APPROACH 2: RAG (LLM + Academic Papers)")
print("-" * 50)
rag_result = ask_with_rag(your_question, preferred_model=your_preferred_model)
print(f"Papers found: {rag_result['num_papers']}")
print(f"Model used: {rag_result['model_used']}")
print(f"Success: {rag_result['success']}")
print(f"Response: {rag_result['response']}")

print("\n" + "=" * 80)

# Approach 3: GraphRAG (LLM + Papers + Knowledge Graph)
print("\n🌐 APPROACH 3: GraphRAG (LLM + Papers + Knowledge Relationships)")
print("-" * 50)
graph_result = ask_with_graph_rag(your_question, preferred_model=your_preferred_model)
print(f"Papers: {len(graph_result['papers_used'])}, Related concepts: {len(graph_result['related_concepts'])}")
print(f"Model used: {graph_result['model_used']}")
print(f"Success: {graph_result['success']}")
print(f"Response: {graph_result['response']}")

print("\n" + "=" * 80)
print("\n🎯 COMPREHENSIVE COMPARISON:")
print("✅ Plain LLM: Fast, general knowledge, no sources")
print("✅ RAG: Cites real papers, focused on direct matches")
print("✅ GraphRAG: Discovers connections, broader context, unexpected insights")
print("\n🔧 TECHNICAL INSIGHTS:")
print(f"✅ All used same model: {plain_result['model_used'] == rag_result['model_used'] == graph_result['model_used']}")
print(f"✅ Fallback worked: {all([r['success'] for r in [plain_result, rag_result, graph_result]])}")
print("✅ Different contexts = different reasoning patterns")


## 🎓 Part 8: Model Selection & Fallback Mechanisms (8-10 minutes)

### 🔧 Why Fallback Mechanisms Matter

**Real-world challenges you'll face:**
- ✅ **Model unavailability**: Your preferred model might be down
- ✅ **Rate limits**: Too many requests = temporary blocks
- ✅ **Cost management**: Free models run out, paid models cost money
- ✅ **Performance variation**: Different models excel at different tasks

**What our fallback system does:**
1. **Tries your preferred model first** (if specified)
2. **Falls back to alternatives** if the first fails
3. **Reports which model actually worked** (transparency)
4. **Handles errors gracefully** (no crashes)

### 🎯 Approach Decision Framework

**Use Plain LLM when:**
- ✅ You need quick, general answers
- ✅ The question is basic/introductory
- ✅ You don't need citations

**Use RAG when:**
- ✅ You need scholarly sources
- ✅ You want verifiable information
- ✅ You're doing focused research on a specific topic
- ✅ You need citations for academic work

**Use GraphRAG when:**
- ✅ You want to discover unexpected connections
- ✅ You're exploring how ideas relate to each other
- ✅ You need broader context beyond direct matches
- ✅ You're doing interdisciplinary research

### 🤖 Model Selection Tips

**For Philosophy Research:**
- **Creative exploration**: Try larger models (GPT-4, Claude)
- **Citation analysis**: Smaller models work fine (Phi-3, Llama-3.2)
- **Quick testing**: Free models are perfect for experimentation
- **Final papers**: Invest in paid models for quality


## 🎉 Workshop Complete! (5 minutes reflection)

### 🏆 What You've Accomplished

**You now understand and can use three different AI research tools:**

1. ✅ **Plain LLM**: Direct API calls for quick answers
2. ✅ **RAG System**: LLM + academic papers for research
3. ✅ **GraphRAG**: LLM + papers + knowledge relationships

**Key concepts you learned:**
- ✅ **What is RAG** and why it's useful for research
- ✅ **What is a Vector Database** and how it finds similar content
- ✅ **What is a Knowledge Graph** and how it connects ideas
- ✅ **What is GraphRAG** and when it discovers new insights
- ✅ **When to use each approach** for different research needs

### 🚀 Take Your Learning Further

**Ideas for your own research:**
- 🔍 Apply these tools to your thesis topic
- 📚 Build a RAG system for your favorite philosophers
- 🌐 Create knowledge graphs for specific philosophical domains
- 🔄 Compare responses across different questions in your field

**What makes you a better researcher now:**
- ✅ You can quickly explore vast amounts of academic literature
- ✅ You can verify AI responses with actual sources
- ✅ You can discover unexpected connections between ideas
- ✅ You know which tool to use for different research questions

### 🎯 The Big Picture

**Philosophy + AI = Powerful combination**

You started this series learning basic Python. Now you can:
- Build sophisticated AI research tools
- Understand when and how to use different approaches
- Combine human insight with machine capabilities
- Navigate complex knowledge spaces efficiently

**This is just the beginning of what's possible when philosophy meets AI!** 🌟

---

### 💡 Final Reflection Questions

1. **Which approach gave you the most surprising insights?**
2. **How might these tools change philosophical research?**
3. **What ethical considerations should guide their use?**
4. **What would you build next?**

**Thank you for joining this journey from simple programming to sophisticated AI research tools!** 🎓
